# Session 2: bones rotate about joints, and joints chain
**A. Real-world problem.** When you wave, your hand travels because the forearm rides on
the elbow, which rides on the shoulder. A character rig is exactly that chain. Today you
build a two-bone arm in metres and answer: where is the hand, and can it reach a target?

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
L1, L2 = 0.30, 0.25          # upper arm and forearm lengths in metres

## Step 1: rotate about a joint, not about the origin
Session 1 turned points about (0, 0). A joint is somewhere else. To turn about a pivot `c`:
move the pivot to the origin, turn, move back:  new = R (x - c) + c.

In [ ]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def rotate_about(points, theta, pivot):
    pivot = np.asarray(pivot, dtype=float)
    return (np.asarray(points, dtype=float) - pivot) @ rot(theta).T + pivot

## Step 2: chain two bones
The upper arm points at angle `a` from the shoulder. The forearm points at `a + b`, because
the elbow angle `b` is measured *relative to* the upper arm. Angles add down the chain:

    elbow = L1 (cos a, sin a)
    hand  = elbow + L2 (cos(a+b), sin(a+b))

In [ ]:
def arm_joints(a, b, l1=L1, l2=L2):
    """Shoulder, elbow, hand as three (x, y) points in metres. Angles in radians."""
    shoulder = np.array([0.0, 0.0])
    elbow = l1 * np.array([np.cos(a), np.sin(a)])
    hand = elbow + l2 * np.array([np.cos(a + b), np.sin(a + b)])
    return np.array([shoulder, elbow, hand])

## Step 3: how far can the hand reach?
The distance from the shoulder to the hand depends only on the elbow angle (law of cosines):
    |hand|^2 = L1^2 + L2^2 + 2 L1 L2 cos b
Straight arm (b = 0) is the longest reach L1 + L2; folded back (b = 180 degrees) is the
shortest, |L1 - L2|.

In [ ]:
def reach(b, l1=L1, l2=L2):
    return np.sqrt(l1 ** 2 + l2 ** 2 + 2 * l1 * l2 * np.cos(b))

## Step 4: run it backwards (design)
Given a target, which angles put the hand there? From the reach formula get `b`, then `a`.
There are two answers: "elbow up" and "elbow down".

In [ ]:
def solve_reach(x, y, l1=L1, l2=L2):
    """Return the two (a, b) solutions in radians, or [] if the target cannot be reached."""
    d2 = x * x + y * y
    cb = (d2 - l1 ** 2 - l2 ** 2) / (2 * l1 * l2)
    if abs(cb) > 1 + 1e-9:            # small tolerance so exact boundary targets are accepted
        return []
    b0 = float(np.arccos(min(1.0, max(-1.0, cb))))
    bs = [b0] if abs(np.sin(b0)) < 1e-9 else [b0, -b0]   # fully stretched or folded: only one solution
    return [(float(np.arctan2(y, x) - np.arctan2(l2 * np.sin(b), l1 + l2 * np.cos(b))), b) for b in bs]

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for a_d, b_d in [(0, 0), (40, 50), (90, -90)]:
        j = arm_joints(np.radians(a_d), np.radians(b_d))
        ax.plot(j[:, 0], j[:, 1], "o-", lw=4, label=f"a={a_d}, b={b_d} deg")
        print(f"a={a_d:3d} deg, b={b_d:4d} deg -> hand {j[2].round(3)} m, reach {reach(np.radians(b_d)):.3f} m")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("Two bones, two angles")
    fig.savefig(OUT / "two_bones.png", dpi=120)

    b_deg = np.linspace(0, 180, 181)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(b_deg, reach(np.radians(b_deg)))
    ax2.axhline(L1 + L2, color="gray", ls=":")
    ax2.axhline(abs(L1 - L2), color="gray", ls=":")
    ax2.set_xlabel("elbow angle b (degrees)")
    ax2.set_ylabel("shoulder-to-hand distance (m)")
    ax2.set_title("Reach: 0.55 m when straight, 0.05 m when folded")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "reach_vs_elbow.png", dpi=120)
    for a, b in solve_reach(0.35, 0.20):
        print(f"target (0.35, 0.20) m: a={np.degrees(a):.2f} deg, b={np.degrees(b):.2f} deg")

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Print the hand position for a = 90 degrees, b = -90 degrees. Predict it first, then compare with the answer to question type B in Section F.
2. **Stretch.** Add a third bone (a hand of length 0.10 m, angle c relative to the forearm) to `arm_joints` and print the fingertip for a = 40, b = 50, c = -30 degrees.
3. **Explain.** Explain why most targets have two solutions and the target (1.0, 0.0) m has none.

## Assessment
- **A. Conceptual.** Why does the forearm point at angle a + b rather than just b?
- **B. Numerical.** For L1 = 0.30 m, L2 = 0.25 m, a = 90 degrees and b = -90 degrees, find the hand position and its distance from the shoulder.
- **C. Graph and simulation interpretation.** From the reach graph, what are the largest and smallest shoulder-to-hand distances, and at which elbow angles?
- **D. Design and optimisation challenge.** Design: find joint angles that put the hand on (0.35, 0.20) m. How many solutions are there? Is (1.0, 0.0) m reachable?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    q = L.rotate_about(np.array([[2.0, 1.0]]), np.pi / 2, [1.0, 1.0])
    checks.append(("rotating (2, 1) by 90 degrees about (1, 1) gives (1, 2)", np.allclose(q, [[1.0, 2.0]], atol=1e-9)))
    j = L.arm_joints(0.0, 0.0)
    checks.append(("straight arm reaches 0.55 m along x", np.allclose(j[2], [0.55, 0.0], atol=1e-9)))
    j = L.arm_joints(np.radians(40), np.radians(50))
    checks.append(("worked example: hand at (0.230, 0.443) m", np.allclose(j[2], [0.2298, 0.4428], atol=1e-3)))
    checks.append(("reach formula agrees with the geometry",
                   abs(np.linalg.norm(j[2]) - L.reach(np.radians(50))) < 1e-9))
    ok = True
    for a in np.linspace(-3, 3, 7):
        for b in np.linspace(-2.5, 2.5, 7):
            jj = L.arm_joints(a, b)
            ok &= abs(np.linalg.norm(jj[1] - jj[0]) - L.L1) < 1e-9
            ok &= abs(np.linalg.norm(jj[2] - jj[1]) - L.L2) < 1e-9
    checks.append(("bone lengths never change", bool(ok)))
    checks.append(("longest reach is L1+L2 and shortest is |L1-L2|",
                   abs(L.reach(0.0) - 0.55) < 1e-9 and abs(L.reach(np.pi) - 0.05) < 1e-9))
    sols = L.solve_reach(0.35, 0.20)
    good = len(sols) == 2 and all(np.allclose(L.arm_joints(a, b)[2], [0.35, 0.20], atol=1e-9) for a, b in sols)
    checks.append(("two ways to reach (0.35, 0.20): elbow up and elbow down", bool(good)))
    checks.append(("a target 1 m away is unreachable", L.solve_reach(1.0, 0.0) == []))
    full = L.solve_reach(0.55, 0.0)
    checks.append(("the fully stretched point (0.55, 0) has exactly one solution, b = 0",
                   len(full) == 1 and abs(full[0][1]) < 1e-6 and abs(full[0][0]) < 1e-6))
    folded = L.solve_reach(0.05, 0.0)
    checks.append(("the fully folded point (0.05, 0) has exactly one solution",
                   len(folded) == 1 and np.allclose(L.arm_joints(*folded[0])[2], [0.05, 0.0], atol=1e-9)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
